In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/ericanacletoribeiro/cicids2017-cleaned-and-preprocessed/cicids2017_cleaned.csv


In [2]:
import pandas as pd

csv_path = "/kaggle/input/datasets/ericanacletoribeiro/cicids2017-cleaned-and-preprocessed/cicids2017_cleaned.csv"

df = pd.read_csv(csv_path)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns)

df.head()

Shape: (2520751, 53)

Columns:
Index(['Destination Port', 'Flow Duration', 'Total Fwd Packets',
       'Total Length of Fwd Packets', 'Fwd Packet Length Max',
       'Fwd Packet Length Min', 'Fwd Packet Length Mean',
       'Fwd Packet Length Std', 'Bwd Packet Length Max',
       'Bwd Packet Length Min', 'Bwd Packet Length Mean',
       'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s',
       'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min',
       'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max',
       'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std',
       'Bwd IAT Max', 'Bwd IAT Min', 'Fwd Header Length', 'Bwd Header Length',
       'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length',
       'Max Packet Length', 'Packet Length Mean', 'Packet Length Std',
       'Packet Length Variance', 'FIN Flag Count', 'PSH Flag Count',
       'ACK Flag Count', 'Average Packet Size', 'Subflow Fwd Bytes',
       'Init_Win_bytes_forward', 'Init_W

,Destination Port,Flow Duration,Total Fwd Packets,Total Length of Fwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,Bwd Packet Length Max,Bwd Packet Length Min,...,Init_Win_bytes_backward,act_data_pkt_fwd,min_seg_size_forward,Active Mean,Active Max,Active Min,Idle Mean,Idle Max,Idle Min,Attack Type
0,22,1266342,41,2664,456,0,64.975610,109.864573,976,0,...,243,24,32,0.0,0,0,0.0,0,0,Normal Traffic
1,22,1319353,41,2664,456,0,64.975610,109.864573,976,0,...,243,24,32,0.0,0,0,0.0,0,0,Normal Traffic
2,22,160,1,0,0,0,0.000000,0.000000,0,0,...,243,0,32,0.0,0,0,0.0,0,0,Normal Traffic
3,22,1303488,41,2728,456,0,66.536585,110.129945,976,0,...,243,24,32,0.0,0,0,0.0,0,0,Normal Traffic
4,35396,77,1,0,0,0,0.000000,0.000000,0,0,...,290,0,32,0.0,0,0,0.0,0,0,Normal Traffic


In [3]:
print(df.columns[-5:])

Index(['Active Min', 'Idle Mean', 'Idle Max', 'Idle Min', 'Attack Type'], dtype='object')


In [4]:
# Separate features and label
X = df.drop(columns=['Attack Type'])
y = df['Attack Type']

print("X shape:", X.shape)
print("y shape:", y.shape)

y.unique()[:10]  # see some label values

X shape: (2520751, 52)
y shape: (2520751,)


array(['Normal Traffic', 'Port Scanning', 'Web Attacks', 'Brute Force',
       'DDoS', 'Bots', 'DoS'], dtype=object)

In [5]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_encoded = le.fit_transform(y)

print("Number of classes:", len(le.classes_))

Number of classes: 7


In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42
)

print(X_train.shape, X_test.shape)

(2016600, 52) (504151, 52)


In [7]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [8]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Use GPU if available, otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

# Convert data to PyTorch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

# DataLoader
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=1024, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1024, shuffle=False)

# Define MLP
class MLP(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        return self.model(x)

input_dim = X_train.shape[1]   # 52
num_classes = len(le.classes_) # 7

mlp = MLP(input_dim, num_classes).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(mlp.parameters(), lr=0.001)

print(mlp)

Using: cpu
MLP(
  (model): Sequential(
    (0): Linear(in_features=52, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): ReLU()
    (5): Linear(in_features=64, out_features=7, bias=True)
  )
)


In [9]:
epochs = 5

for epoch in range(epochs):
    mlp.train()
    total_loss = 0

    for batch_X, batch_y in train_loader:
        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)

        optimizer.zero_grad()
        outputs = mlp(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(train_loader):.4f}")

Epoch 1/5, Loss: 0.1140
Epoch 2/5, Loss: 0.0553
Epoch 3/5, Loss: 0.0467
Epoch 4/5, Loss: 0.0419
Epoch 5/5, Loss: 0.0392


In [10]:
mlp.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch_X, batch_y in test_loader:
        batch_X = batch_X.to(device)

        outputs = mlp(batch_X)
        preds = torch.argmax(outputs, dim=1).cpu().numpy()

        all_preds.extend(preds)
        all_labels.extend(batch_y.numpy())

accuracy = accuracy_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds, average="weighted")

print("Clean MLP Accuracy:", accuracy)
print("Clean MLP Weighted F1:", f1)
print(classification_report(all_labels, all_preds, target_names=le.classes_))

Clean MLP Accuracy: 0.9859585719357891
Clean MLP Weighted F1: 0.9852515410230762
                precision    recall  f1-score   support

          Bots       0.98      0.33      0.49       384
   Brute Force       0.92      0.96      0.94      1821
          DDoS       1.00      1.00      1.00     25675
           DoS       0.98      0.98      0.98     38912
Normal Traffic       0.99      0.99      0.99    418697
 Port Scanning       0.90      0.83      0.86     18219
   Web Attacks       1.00      0.04      0.07       443

      accuracy                           0.99    504151
     macro avg       0.97      0.73      0.76    504151
  weighted avg       0.99      0.99      0.99    504151



In [11]:
def fgsm_attack(model, X, y, epsilon):
    X_adv = X.clone().detach().to(device)
    y = y.to(device)

    X_adv.requires_grad = True

    outputs = model(X_adv)
    loss = criterion(outputs, y)

    model.zero_grad()
    loss.backward()

    # FGSM perturbation
    perturbation = epsilon * X_adv.grad.sign()
    X_adv = X_adv + perturbation

    return X_adv.detach()

In [12]:
epsilon = 0.05

mlp.eval()
fgsm_preds = []
fgsm_labels = []

adv_batches = []
label_batches = []

for batch_X, batch_y in test_loader:
    batch_X = batch_X.to(device)
    batch_y = batch_y.to(device)

    adv_X = fgsm_attack(mlp, batch_X, batch_y, epsilon)

    # save full adversarial dataset
    adv_batches.append(adv_X.cpu())
    label_batches.append(batch_y.cpu())

    # evaluate MLP on adversarial data
    with torch.no_grad():
        outputs = mlp(adv_X)
        preds = torch.argmax(outputs, dim=1).cpu().numpy()

    fgsm_preds.extend(preds)
    fgsm_labels.extend(batch_y.cpu().numpy())

X_test_fgsm = torch.cat(adv_batches).numpy()
y_test_fgsm = torch.cat(label_batches).numpy()

fgsm_acc = accuracy_score(fgsm_labels, fgsm_preds)
fgsm_f1 = f1_score(fgsm_labels, fgsm_preds, average="weighted")

print("FGSM Epsilon:", epsilon)
print("X_test_fgsm shape:", X_test_fgsm.shape)
print("FGSM Accuracy:", fgsm_acc)
print("FGSM Weighted F1:", fgsm_f1)
print("Accuracy Drop:", accuracy - fgsm_acc)
print("F1 Drop:", f1 - fgsm_f1)
print(classification_report(fgsm_labels, fgsm_preds, target_names=le.classes_))

FGSM Epsilon: 0.05
X_test_fgsm shape: (504151, 52)
FGSM Accuracy: 0.8708363168971202
FGSM Weighted F1: 0.8714581871584695
Accuracy Drop: 0.11512225503866891
F1 Drop: 0.11379335386460676
                precision    recall  f1-score   support

          Bots       0.00      0.00      0.00       384
   Brute Force       0.04      0.02      0.03      1821
          DDoS       0.42      0.55      0.47     25675
           DoS       0.82      0.86      0.84     38912
Normal Traffic       0.93      0.92      0.92    418697
 Port Scanning       0.50      0.36      0.41     18219
   Web Attacks       1.00      0.00      0.00       443

      accuracy                           0.87    504151
     macro avg       0.53      0.39      0.38    504151
  weighted avg       0.87      0.87      0.87    504151



In [13]:
def pgd_attack(model, X, y, epsilon=0.05, alpha=0.01, num_iter=10):
    X_original = X.clone().detach().to(device)
    y = y.to(device)

    # Start from the original input
    X_adv = X_original.clone().detach()

    for _ in range(num_iter):
        X_adv.requires_grad = True

        outputs = model(X_adv)
        loss = criterion(outputs, y)

        model.zero_grad()
        loss.backward()

        # Small FGSM-like step
        X_adv = X_adv + alpha * X_adv.grad.sign()

        # Project back into epsilon-ball around original input
        perturbation = torch.clamp(X_adv - X_original, min=-epsilon, max=epsilon)
        X_adv = torch.clamp(X_original + perturbation, min=X_train.min(), max=X_train.max()).detach()

    return X_adv

In [14]:
epsilon = 0.05
alpha = 0.01
num_iter = 10

mlp.eval()
pgd_preds = []
pgd_labels = []

pgd_batches = []
pgd_label_batches = []

for batch_X, batch_y in test_loader:
    batch_X = batch_X.to(device)
    batch_y = batch_y.to(device)

    adv_X = pgd_attack(mlp, batch_X, batch_y, epsilon, alpha, num_iter)

    pgd_batches.append(adv_X.cpu())
    pgd_label_batches.append(batch_y.cpu())

    with torch.no_grad():
        outputs = mlp(adv_X)
        preds = torch.argmax(outputs, dim=1).cpu().numpy()

    pgd_preds.extend(preds)
    pgd_labels.extend(batch_y.cpu().numpy())

X_test_pgd = torch.cat(pgd_batches).numpy()
y_test_pgd = torch.cat(pgd_label_batches).numpy()

pgd_acc = accuracy_score(pgd_labels, pgd_preds)
pgd_f1 = f1_score(pgd_labels, pgd_preds, average="weighted")

print("PGD Epsilon:", epsilon)
print("PGD Alpha:", alpha)
print("PGD Iterations:", num_iter)
print("X_test_pgd shape:", X_test_pgd.shape)
print("PGD Accuracy:", pgd_acc)
print("PGD Weighted F1:", pgd_f1)
print("Accuracy Drop:", accuracy - pgd_acc)
print("F1 Drop:", f1 - pgd_f1)

PGD Epsilon: 0.05
PGD Alpha: 0.01
PGD Iterations: 10
X_test_pgd shape: (504151, 52)
PGD Accuracy: 0.8507768505864315
PGD Weighted F1: 0.8506311137151426
Accuracy Drop: 0.13518172134935758
F1 Drop: 0.13462042730793367


In [15]:
torch.save(mlp.state_dict(), "/kaggle/working/mlp_model.pth")

In [16]:
import json

results = {
    "clean_accuracy": accuracy,
    "clean_f1": f1,
    "fgsm_accuracy": fgsm_acc,
    "fgsm_f1": fgsm_f1,
    "pgd_accuracy": pgd_acc,
    "pgd_f1": pgd_f1
}

with open("/kaggle/working/results.json", "w") as f:
    json.dump(results, f)

In [17]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

# Use smaller training subset so it runs faster
N = 300000
X_train_small = X_train[:N]
y_train_small = y_train[:N]

lr_model = LogisticRegression(max_iter=500, n_jobs=-1)
lr_model.fit(X_train_small, y_train_small)

rf_model = RandomForestClassifier(
    n_estimators=50,
    max_depth=20,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train_small, y_train_small)

for name, model in [("LR", lr_model), ("RF", rf_model)]:
    preds = model.predict(X_test)
    print(name, "Clean Accuracy:", accuracy_score(y_test, preds))
    print(name, "Clean F1:", f1_score(y_test, preds, average="weighted"))

LR Clean Accuracy: 0.9740534086017879
LR Clean F1: 0.9741312474878479
RF Clean Accuracy: 0.9982346558868276
RF Clean F1: 0.9981791309729047


In [18]:
print(X_test_fgsm.shape)
print(X_test.shape)

(504151, 52)
(504151, 52)


In [19]:
for name, model in [("LR", lr_model), ("RF", rf_model)]:
    preds = model.predict(X_test_fgsm)
    print(name, "FGSM Accuracy:", accuracy_score(y_test_fgsm, preds))
    print(name, "FGSM F1:", f1_score(y_test_fgsm, preds, average="weighted"))

LR FGSM Accuracy: 0.8573522615248209
LR FGSM F1: 0.8512331868835658
RF FGSM Accuracy: 0.8404446286925941
RF FGSM F1: 0.7769090589850491


In [20]:
print(X_test_pgd.shape)
print(X_test.shape)

(504151, 52)
(504151, 52)


In [21]:
for name, model in [("LR", lr_model), ("RF", rf_model)]:
    preds = model.predict(X_test_pgd)
    print(name, "PGD Accuracy:", accuracy_score(y_test_pgd, preds))
    print(name, "PGD F1:", f1_score(y_test_pgd, preds, average="weighted"))

LR PGD Accuracy: 0.8463119184530031
LR PGD F1: 0.8414443548881352
RF PGD Accuracy: 0.8454828017796255
RF PGD F1: 0.7859115097953046
